# Bounded Grover search

Retain both two-bit values, the complete predicate and two Grover iterations. Width is the declared synthesis objective. Seeded local samples from exact CPU probabilities replace cloud sampling. Named x/y outputs replace the source structure fields; the predicate is evaluated on sampled rows. Run all cells from the repository root. See WORKFLOW.md for acceptance and adaptations.


In [ ]:
from math import pi
from typing import Annotated

import flagquantum as fq
from flagquantum import synth
from flagquantum.synth import (
    Output,
    QNum,
    allocate,
    control,
    hadamard_transform,
    phase,
    qfunc,
)


@qfunc
def mark(x: Annotated[QNum, 2], y: Annotated[QNum, 2]) -> None:
    control((x + y < 4) & (((x * y) % 4) == 2), lambda: phase(1, pi))


@qfunc
def diffuse(x: Annotated[QNum, 2], y: Annotated[QNum, 2]) -> None:
    hadamard_transform(x)
    hadamard_transform(y)
    control((x < 1) & (y < 1), lambda: phase(1, pi))
    phase(1, pi)
    hadamard_transform(x)
    hadamard_transform(y)


@qfunc
def main(x: Output[Annotated[QNum, 2]], y: Output[Annotated[QNum, 2]]) -> None:
    allocate(x)
    allocate(y)
    hadamard_transform(x)
    hadamard_transform(y)
    for _ in range(2):
        mark(x, y)
        diffuse(x, y)



In [ ]:
program = synth.synthesize(main, objective="width")
result = fq.run(program)
probabilities = dict(program.probabilities(result, registers=("x", "y")))
print(probabilities)
print(program.report["resources"])


In [ ]:
from collections import Counter
from random import Random

counts = Counter(Random(17).choices(list(probabilities), weights=list(probabilities.values()), k=1000))
rows = [
    {"x": x, "y": y, "counts": count, "predicate": x + y < 4 and (x * y) % 4 == 2}
    for (x, y), count in sorted(counts.items())
]
assert sum(row["counts"] for row in rows) == 1000
print(rows)
